# Dark Manifold Virtual Cell — validation against Breuer 2019

Loads the trained `dark_manifold_100gene.pt` checkpoint from `Nikku03/cell-simulation` and validates it against the JCVI-Syn3A essentiality labels in `Nikku03/cell` (Breuer et al. 2019, eLife).

Closes the validation gap explicitly named in `cell-simulation/GAP_ANALYSIS.md`: **"No validation against experimental essentiality data."**

Method:
1. Clone both repos.
2. Load `DarkManifold100Gene` checkpoint.
3. For each of the 102 genes the model knows, run a knockout via `enzyme_mask[gene_idx] = 0`, roll out a trajectory, derive an essentiality score from the predicted ATP / energy-charge drop.
4. Map gene names (`pyk`, `rpoA`, ...) → Syn3A locus tags (`JCVISYN3A_xxxx`) via the GenBank `/gene` qualifier.
5. Join with the 455 Breuer 2019 essentiality labels in `labels.csv`. Compute MCC at the optimal threshold and report.

Realistic expectation per the existing `GAP_ANALYSIS.md`: modest MCC (~0.2–0.4). Hub-gene wrong-sign errors and FBA-synthetic training data bound the ceiling. The number itself is the deliverable — a **measured** validation against real experimental labels.

Runs on Colab (CPU is enough; the model is ~1 MB).

---

## Cell 1 — clone repos + install deps

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Clone Dark Manifold model repo
DM_REPO = Path('/content/cell-simulation')
if not DM_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Nikku03/cell-simulation.git',
                    str(DM_REPO)], check=True)
print(f'Dark Manifold repo: {DM_REPO}')

# Clone the Syn3A simulator repo (for Breuer labels)
CELL_REPO = Path('/content/cell')
if not CELL_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    '-b', 'claude/syn3a-whole-cell-simulator-REjHC',
                    'https://github.com/Nikku03/cell.git',
                    str(CELL_REPO)], check=True)
print(f'Cell repo: {CELL_REPO}')

# Clone the Luthey-Schulten Minimal_Cell_ComplexFormation repo INTO the cell
# tree so the existing path cell_sim/data/Minimal_Cell_ComplexFormation/...
# resolves. The cell repo intentionally does NOT commit this third-party
# data — users clone it separately via cell_sim/setup.sh in normal use.
MCCF = CELL_REPO / 'cell_sim/data/Minimal_Cell_ComplexFormation'
if not (MCCF / 'input_data/syn3A.gb').exists():
    # Remove any empty placeholder dir, then clone
    if MCCF.exists():
        import shutil; shutil.rmtree(MCCF)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation.git',
                    str(MCCF)], check=True)
assert (MCCF / 'input_data/syn3A.gb').exists(), 'Syn3A GenBank still missing'
print(f'Luthey-Schulten data: {MCCF}')

# Install deps (torch should already be on Colab; biopython for GenBank)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'biopython', 'pandas', 'numpy', 'scikit-learn'],
               check=True)

sys.path.insert(0, str(DM_REPO))
print('OK')

## Cell 2 — load the Dark Manifold checkpoint

The checkpoint is a dict with `model_state_dict`, `genes` (list of 102 names), `metabolites` (list of 74), and `results` (already-computed quick-eval results).

In [ ]:
import torch, json, numpy as np

ckpt = torch.load(DM_REPO / 'dark_manifold_100gene.pt',
                  map_location='cpu', weights_only=False)
DM_GENES = ckpt['genes']
DM_METS  = ckpt['metabolites']
n_genes = len(DM_GENES)
n_mets  = len(DM_METS)
print(f'genes: {n_genes}, metabolites: {n_mets}')
print(f'sample genes: {DM_GENES[:8]}')
print(f'sample mets:  {DM_METS[:8]}')
print()
print('checkpoint summary:')
for k, v in ckpt['results'].get('summary', {}).items():
    print(f'  {k}: {v}')

# Instantiate the model architecture and load weights
from dark_manifold_100gene import DarkManifold100Gene
model = DarkManifold100Gene(n_genes=n_genes, n_mets=n_mets, hidden_dim=256)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'\nmodel parameters: {sum(p.numel() for p in model.parameters()):,}')

## Cell 3 — knockout sweep

For each of 102 genes, run a knockout via `enzyme_mask[gene_idx] = 0` and roll out a trajectory. Extract two essentiality signals:

* **ATP drop** at terminal step — `atp_t_final − atp_wt_final`
* **Energy charge drop** — `EC_t_final − EC_wt_final` where `EC = (ATP + 0.5·ADP) / (ATP + ADP + AMP)`

A gene is predicted essential if `|atp_drop|` exceeds a threshold. We sweep thresholds in cell 5 to find the optimal MCC.

In [ ]:
import torch
import numpy as np
import time

ROLLOUT_STEPS = 100
ATP_IDX  = DM_METS.index('ATP') if 'ATP' in DM_METS else None
ADP_IDX  = DM_METS.index('ADP') if 'ADP' in DM_METS else None
AMP_IDX  = DM_METS.index('AMP') if 'AMP' in DM_METS else None
print(f'ATP idx: {ATP_IDX}, ADP idx: {ADP_IDX}, AMP idx: {AMP_IDX}')

def initial_state(batch=1):
    """Reference state: enzymes ~ 1.0, metabolites ~ 1.0, ATP/ADP/AMP at physiological ratio."""
    state = torch.ones(batch, n_genes + n_mets) * 1.0
    if ATP_IDX is not None: state[:, n_genes + ATP_IDX] = 3.0
    if ADP_IDX is not None: state[:, n_genes + ADP_IDX] = 1.0
    if AMP_IDX is not None: state[:, n_genes + AMP_IDX] = 0.3
    return state

def rollout(model, state, enzyme_mask, n_steps):
    s = state.clone()
    for _ in range(n_steps):
        new_state = model(s, enzyme_mask=enzyme_mask)
        # The forward returns a state tensor of same shape (per architecture).
        # If the architecture's forward returns a dict, adapt:
        if isinstance(new_state, dict):
            new_state = new_state.get('state', new_state.get('next_state', new_state))
        s = new_state
    return s

# Wild-type baseline
with torch.no_grad():
    wt_mask = torch.ones(1, n_genes)
    s0 = initial_state()
    wt_final = rollout(model, s0, wt_mask, ROLLOUT_STEPS)
    wt_atp = wt_final[0, n_genes + ATP_IDX].item() if ATP_IDX is not None else 0.0
    wt_adp = wt_final[0, n_genes + ADP_IDX].item() if ADP_IDX is not None else 0.0
    wt_amp = wt_final[0, n_genes + AMP_IDX].item() if AMP_IDX is not None else 0.0
    wt_ec = (wt_atp + 0.5 * wt_adp) / max(wt_atp + wt_adp + wt_amp, 1e-9)
    print(f'WT terminal: ATP={wt_atp:.3f}  ADP={wt_adp:.3f}  AMP={wt_amp:.3f}  EC={wt_ec:.3f}')

# Per-gene KO
rows = []
t0 = time.time()
with torch.no_grad():
    for i, gene in enumerate(DM_GENES):
        mask = torch.ones(1, n_genes)
        mask[0, i] = 0.0
        ko_final = rollout(model, initial_state(), mask, ROLLOUT_STEPS)
        atp = ko_final[0, n_genes + ATP_IDX].item() if ATP_IDX is not None else 0.0
        adp = ko_final[0, n_genes + ADP_IDX].item() if ADP_IDX is not None else 0.0
        amp = ko_final[0, n_genes + AMP_IDX].item() if AMP_IDX is not None else 0.0
        ec  = (atp + 0.5 * adp) / max(atp + adp + amp, 1e-9)
        rows.append({
            'gene': gene,
            'ko_atp': atp,
            'ko_ec':  ec,
            'atp_drop':  atp - wt_atp,
            'ec_drop':   ec - wt_ec,
            'abs_atp_drop': abs(atp - wt_atp),
            'abs_ec_drop':  abs(ec - wt_ec),
        })
elapsed = time.time() - t0
print(f'\nknockout sweep: {len(rows)} genes in {elapsed:.1f}s')
import pandas as pd
df_ko = pd.DataFrame(rows)
df_ko.head(8)

## Cell 4 — map Dark Manifold gene names → Syn3A locus tags

The Dark Manifold uses gene names like `pyk`. The Breuer 2019 labels are keyed by `JCVISYN3A_xxxx` locus tags. The Syn3A GenBank file has both.

In [ ]:
from Bio import SeqIO
import pandas as pd

GB_PATH = CELL_REPO / 'cell_sim/data/Minimal_Cell_ComplexFormation/input_data/syn3A.gb'
rec = next(SeqIO.parse(str(GB_PATH), 'genbank'))

name_to_locus = {}
for f in rec.features:
    if f.type != 'CDS':
        continue
    locus = f.qualifiers.get('locus_tag', [None])[0]
    gene  = f.qualifiers.get('gene', [None])[0]
    if locus and gene:
        name_to_locus[gene.lower()] = locus

print(f'Syn3A gene-name -> locus-tag mapping: {len(name_to_locus)} entries')

# Map Dark Manifold genes
df_ko['gene_lower'] = df_ko['gene'].str.lower()
df_ko['locus_tag'] = df_ko['gene_lower'].map(name_to_locus)
matched = df_ko['locus_tag'].notna().sum()
print(f'  Dark Manifold -> Syn3A locus matched: {matched} / {len(df_ko)}')
df_ko[['gene', 'locus_tag']].head(15)

## Cell 5 — join with Breuer 2019, sweep thresholds, compute MCC

In [ ]:
from sklearn.metrics import matthews_corrcoef, confusion_matrix
import pandas as pd, numpy as np

LABELS_CSV = CELL_REPO / 'memory_bank/data/multiorg_essentiality/labels.csv'
labels = pd.read_csv(LABELS_CSV)
labels = labels[labels.organism == 'syn3a'][['locus_tag', 'essential']].rename(
    columns={'essential': 'gt_essential'})
print(f'Breuer 2019 Syn3A labels: {len(labels)} ({int(labels.gt_essential.sum())} essential)')

joined = df_ko.merge(labels, on='locus_tag', how='inner')
print(f'joined (DM x Breuer): {len(joined)} genes')
print()

# Sweep |atp_drop| threshold and find the threshold that maximises MCC
results = []
thresholds = sorted(set(np.linspace(0, max(joined.abs_atp_drop) * 1.01, 200)) |
                     set(joined.abs_atp_drop.tolist()))
for thr in thresholds:
    pred = (joined.abs_atp_drop >= thr).astype(int)
    if pred.sum() == 0 or pred.sum() == len(pred):
        mcc = 0.0
    else:
        mcc = matthews_corrcoef(joined.gt_essential, pred)
    results.append({'threshold': thr, 'n_pred_essential': int(pred.sum()), 'mcc': mcc})
df_thr = pd.DataFrame(results).sort_values('mcc', ascending=False)
best = df_thr.iloc[0]
print('Best |atp_drop| threshold for MCC:')
print(f'  threshold = {best.threshold:.4f}')
print(f'  predicted essential: {best.n_pred_essential}')
print(f'  MCC = {best.mcc:.4f}')
print()

# Final confusion matrix at best threshold
best_pred = (joined.abs_atp_drop >= best.threshold).astype(int)
cm = confusion_matrix(joined.gt_essential, best_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f'  TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'  precision = {tp / max(tp+fp, 1):.3f}')
print(f'  recall    = {tp / max(tp+fn, 1):.3f}')
print(f'  specificity = {tn / max(tn+fp, 1):.3f}')
print()
print('Reference baselines for context:')
print('  v15 (biology-first detector, full Syn3A panel):  MCC=0.5372')
print('  v18 (cross-organism XGBoost, Syn3A held out):    MCC=0.1376')
print('  Dark Manifold (this measurement):                 MCC={:.4f}'.format(best.mcc))

## Cell 6 — same sweep using `|ec_drop|` (energy-charge proxy)

In [ ]:
import numpy as np
from sklearn.metrics import matthews_corrcoef

results_ec = []
for thr in sorted(set(np.linspace(0, max(joined.abs_ec_drop) * 1.01, 200)) |
                   set(joined.abs_ec_drop.tolist())):
    pred = (joined.abs_ec_drop >= thr).astype(int)
    if pred.sum() == 0 or pred.sum() == len(pred):
        mcc = 0.0
    else:
        mcc = matthews_corrcoef(joined.gt_essential, pred)
    results_ec.append({'threshold': thr, 'n_pred_essential': int(pred.sum()), 'mcc': mcc})
df_ec = pd.DataFrame(results_ec).sort_values('mcc', ascending=False)
print('Best |ec_drop| threshold for MCC:')
print(df_ec.head(5).to_string(index=False))
best_ec = df_ec.iloc[0]
print(f'\nBest MCC by EC drop: {best_ec.mcc:.4f}')

## Cell 7 — write results to /content/cell repo for committing back

This dumps a JSON the user can pull and commit as a memory_bank fact.

In [ ]:
import json

out = {
    'method': 'dark_manifold_100gene_knockout_sweep',
    'rollout_steps': ROLLOUT_STEPS,
    'n_dm_genes': int(n_genes),
    'n_dm_mets':  int(n_mets),
    'n_genes_mapped_to_syn3a': int(matched),
    'n_genes_with_breuer_label': int(len(joined)),
    'breuer_essential_in_subset': int(joined.gt_essential.sum()),
    'breuer_nonessential_in_subset': int((1 - joined.gt_essential).sum()),
    'best_atp_drop': {
        'threshold': float(best.threshold),
        'mcc': float(best.mcc),
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        'precision': float(tp / max(tp+fp, 1)),
        'recall':    float(tp / max(tp+fn, 1)),
    },
    'best_ec_drop': {
        'threshold': float(best_ec.threshold),
        'mcc':       float(best_ec.mcc),
    },
    'baselines': {
        'v15_syn3a': 0.5372,
        'v18_xgboost_syn3a_held_out': 0.1376,
    },
    'caveats': [
        'Dark Manifold trained on FBA-synthetic ground truth, not measured biology',
        'GAP_ANALYSIS.md flags hub-gene wrong-sign predictions (groEL, rpoB, dnaA)',
        'Threshold tuned post-hoc on the same data; report best-MCC at any threshold',
        f'Only {int(matched)} of {int(n_genes)} DM genes map cleanly to Syn3A locus tags via /gene field',
    ],
}
out_path = CELL_REPO / 'outputs/dark_manifold_validation.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'wrote {out_path}')
print()
print(json.dumps(out, indent=2))

## Cell 8 — push results back to the branch (optional)

If you set a GitHub PAT, this commits the JSON output back to the dev branch.

In [ ]:
import os, subprocess

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
if not GITHUB_TOKEN:
    print('Skip: set GITHUB_TOKEN in Colab secrets to push results back.')
else:
    os.chdir(CELL_REPO)
    subprocess.run(['git', 'config', 'user.email', 'colab@noreply.local'], check=True)
    subprocess.run(['git', 'config', 'user.name',  'Colab Validation'], check=True)
    subprocess.run(['git', 'remote', 'set-url', 'origin',
                    f'https://{GITHUB_TOKEN}@github.com/Nikku03/cell.git'], check=True)
    subprocess.run(['git', 'add', 'outputs/dark_manifold_validation.json'], check=True)
    subprocess.run(['git', 'commit', '-m',
                    'data: Dark Manifold validation against Breuer 2019 (Colab notebook output)'],
                   check=True)
    subprocess.run(['git', 'push', 'origin',
                    'claude/syn3a-whole-cell-simulator-REjHC'], check=True)
    print('Pushed to claude/syn3a-whole-cell-simulator-REjHC')